# CA2: Global Digital Transformation Readiness
**A Longitudinal Statistical Assessment of Digital Infrastructure and Technology Development Across Countries (2010–2024)**

* **Student Name / ID:** Jose Guadalupe Briones Acevedo (2026739)
* **Module:** Statistical Techniques for Data Analysis
* **Institution:** CCT College Dublin
* **GitHub Repository:** https://github.com/2026739-Jose/CA-1-Capstone-Project_Stats-for-Data-Analytics/tree/main/CA2
* **Video Presentation (Google Drive):** *Link*

## 1. Research Design

### 1.1 Project Identity and Business Problem
* **Domain:** Technology & Digital Transformation (Project Management Perspective)
* **Scope & Unit of Analysis:** Global country-year longitudinal panel (`2010–2024`)
* **Primary Data Source:** World Bank – World Development Indicators (WDI), observational data
* **Analytical Approach:** Descriptive statistics, confidence intervals, parametric and non-parametric hypothesis testing, and panel regression analysis in Python.

**Business Problem:**  
International organizations planning digital transformation initiatives across multiple countries face substantial financial, operational, and adoption risks when entering markets with underdeveloped technological foundations. From a **Project Management** standpoint, committing capital without assessing national digital readiness increases the risk of project delays, cost overruns, and low user adoption. This study evaluates how digital infrastructure and technological readiness have evolved across countries and geographic regions between 2010 and 2024, providing empirical evidence to support market prioritization and risk mitigation strategies—rather than claiming any single country is universally optimal.

### 1.2 Research Question
> *How has digital infrastructure readiness evolved across countries and regions, and which economic and technological factors are associated with its development over time?*

### 1.3 Specific Objectives
1. **Describe** the longitudinal evolution of key digital infrastructure and technology indicators across countries from 2010 to 2024.
2. **Compare** technological progress and infrastructure gaps across geographic regions and five-year periods (`2010–2014` vs. `2020–2024`).
3. **Examine** statistically significant differences between time periods and country groups using formal parametric and non-parametric hypothesis tests.
4. **Investigate** statistical associations between digital infrastructure expansion, innovation capacity (R&D), and economic development (GDP per capita).
5. **Translate** empirical findings into actionable Project Management recommendations for planning, prioritizing, and de-risking international digital transformation projects.

### 1.4 General Research Hypotheses
To guide the subsequent statistical testing phase (once data coverage is audited), this study formulates five core hypotheses:
* **H1 (Temporal Evolution):** Digital adoption (internet penetration) and connectivity (fixed broadband) increased significantly between the initial period (`2010–2014`) and the recent period (`2020–2024`).
* **H2 (Regional Disparities):** The magnitude of digital infrastructure growth differs significantly across global geographic regions.
* **H3 (Categorical Independence):** A country's geographic region is statistically associated with its level of digital improvement (e.g., low, moderate, or high growth tier).
* **H4 (Innovation Association):** Gains in digital connectivity are positively associated with changes in national Research and Development (R&D) expenditure.
* **H5 (Economic & Structural Drivers):** Within-country improvements in digital infrastructure over time are significantly associated with economic development (GDP per capita) and ICT sector activity, after controlling for country-specific and time-specific effects.

## 2. Data Source Verification & Initial Audit

### 2.1 Official Source and Indicators
All primary indicators and country metadata are retrieved programmatically from the **World Bank World Development Indicators (WDI) API v2** (`https://api.worldbank.org/v2`), which provides open access without authentication. To ensure reproducibility, raw API responses are saved in `data/raw/` before any filtering or aggregation.

* **Numeric Indicators (`2010–2024`):**
  * `IT.NET.USER.ZS`: Individuals using the Internet (% of population) — *Digital adoption*
  * `IT.NET.BBND.P2`: Fixed broadband subscriptions (per 100 people) — *Connectivity infrastructure*
  * `IT.NET.SECR.P6`: Secure Internet servers (per 1 million people) — *Secure digital infrastructure*
  * `BX.GSR.CCIS.ZS`: ICT service exports (% of service exports) — *ICT economic participation*
  * `GB.XPD.RSDV.GD.ZS`: Research and development expenditure (% of GDP) — *Innovation capacity*
  * `NY.GDP.PCAP.KD`: GDP per capita (constant 2015 US$) — *Economic context / control*
* **Categorical & Identifier Fields:** `country_code`, `country_name`, `region`, `income_group`, and `year` (obtained from official World Bank metadata, excluding regional/income aggregates).

In [1]:
import os
import numpy as np
import pandas as pd
import requests

# Ensure project directories exist relative to notebooks/
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

BASE_URL = "https://api.worldbank.org/v2"
START_YEAR = 2010
END_YEAR = 2024

INDICATORS = {
    "internet_users": "IT.NET.USER.ZS",
    "fixed_broadband": "IT.NET.BBND.P2",
    "secure_servers": "IT.NET.SECR.P6",
    "ict_exports": "BX.GSR.CCIS.ZS",
    "rd_expenditure": "GB.XPD.RSDV.GD.ZS",
    "gdp_per_capita": "NY.GDP.PCAP.KD",
}


def fetch_country_metadata() -> pd.DataFrame:
  """Fetch official country metadata from World Bank API v2, excluding aggregates."""
  url = f"{BASE_URL}/country"
  params = {"format": "json", "per_page": 500}
  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  payload = response.json()

  records = []
  for item in payload[1]:
    region_info = item.get("region", {})
    income_info = item.get("incomeLevel", {})
    # Filter out non-country aggregates (coded with region id == "NA")
    if region_info.get("id") == "NA":
      continue
    records.append({
        "country_code": item["id"],
        "country_name": item["name"],
        "region": region_info.get("value"),
        "income_group": income_info.get("value"),
    })
  return pd.DataFrame(records).drop_duplicates(subset=["country_code"])


def fetch_indicator_series(alias: str, indicator_code: str) -> pd.DataFrame:
  """Fetch annual indicator values across all entities for the target year range."""
  url = f"{BASE_URL}/country/all/indicator/{indicator_code}"
  params = {
      "date": f"{START_YEAR}:{END_YEAR}",
      "format": "json",
      "per_page": 25000,
  }
  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  payload = response.json()

  rows = [
      {
          "country_code": entry.get("countryiso3code"),
          "year": int(entry["date"]),
          alias: entry.get("value"),
      }
      for entry in payload[1]
      if entry.get("countryiso3code")
  ]
  return pd.DataFrame(rows)


# 1. Download metadata and save raw snapshot
metadata_df = fetch_country_metadata()
metadata_df.to_csv(f"{RAW_DIR}/country_metadata.csv", index=False)

# 2. Download each WDI indicator and merge on country_code and year
series_dfs = [
    fetch_indicator_series(alias, code) for alias, code in INDICATORS.items()
]
panel_raw = series_dfs[0]
for df_part in series_dfs[1:]:
  panel_raw = panel_raw.merge(df_part, on=["country_code", "year"], how="outer")

# Attach official country metadata (inner join keeps individual countries only)
panel_raw = panel_raw.merge(metadata_df, on="country_code", how="inner")
ordered_cols = [
    "country_code",
    "country_name",
    "region",
    "income_group",
    "year",
    *INDICATORS.keys(),
]
panel_raw = (
    panel_raw[ordered_cols]
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)
panel_raw.to_csv(f"{RAW_DIR}/wdi_indicators_raw.csv", index=False)

print(
    f"Countries: {panel_raw['country_code'].nunique()} | Total country-year"
    f" rows: {len(panel_raw)}"
)

# 3. Audit indicator coverage overall and across comparative 5-year periods
early_years = set(range(2010, 2015))
recent_years = set(range(2020, 2025))

audit_rows = []
for col in INDICATORS.keys():
  valid_mask = panel_raw[col].notna()
  non_null_count = int(valid_mask.sum())
  coverage_pct = round((non_null_count / len(panel_raw)) * 100, 2)
  countries_avail = panel_raw.loc[valid_mask, "country_code"].nunique()

  early_countries = set(
      panel_raw.loc[
          valid_mask & panel_raw["year"].isin(early_years), "country_code"
      ]
  )
  recent_countries = set(
      panel_raw.loc[
          valid_mask & panel_raw["year"].isin(recent_years), "country_code"
      ]
  )
  paired_count = len(early_countries & recent_countries)

  audit_rows.append({
      "indicator": col,
      "non_null_rows": non_null_count,
      "coverage_pct": coverage_pct,
      "countries_with_data": countries_avail,
      "paired_countries_2010_14_vs_2020_24": paired_count,
  })

audit_summary = pd.DataFrame(audit_rows)
display(audit_summary)

Countries: 217 | Total country-year rows: 3255


,indicator,non_null_rows,coverage_pct,countries_with_data,paired_countries_2010_14_vs_2020_24
0,internet_users,2894,88.91,209,187
1,fixed_broadband,2900,89.09,208,207
2,secure_servers,3193,98.10,215,213
3,ict_exports,2643,81.20,189,174
4,rd_expenditure,1310,40.25,143,99
5,gdp_per_capita,3122,95.91,213,208
